In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import os
import bs4
import requests
import yfinance as yf

In [2]:
# first scraping wikipedia to get 500 comapnies of S&P500
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
page = requests.get(url, headers=headers)
print(page.status_code)

200


In [3]:
# print(page.content)

In [4]:
soup = bs4.BeautifulSoup(page.content, 'html.parser')
# soup.body

In [5]:
#  syntax: find_all(name, attrs, recursive, string, limit, **kwargs)
all_links = soup.body.find_all(name='a', attrs={'class':'external', 'rel':'nofollow'})
all_links[:6]


[<a class="external text" href="https://www.nyse.com/quote/XNYS:MMM" rel="nofollow">MMM</a>,
 <a class="external text" href="https://www.nyse.com/quote/XNYS:AOS" rel="nofollow">AOS</a>,
 <a class="external text" href="https://www.nyse.com/quote/XNYS:ABT" rel="nofollow">ABT</a>,
 <a class="external text" href="https://www.nyse.com/quote/XNYS:ABBV" rel="nofollow">ABBV</a>,
 <a class="external text" href="https://www.nyse.com/quote/XNYS:ACN" rel="nofollow">ACN</a>,
 <a class="external text" href="https://www.nasdaq.com/market-activity/stocks/adbe" rel="nofollow">ADBE</a>]

In [6]:
import re

def remove_tags(text):
    # r'<.*?>' matches everything between < and > non-greedily
    clean_text = re.sub(r'<.*?>', '', text)
    return clean_text


In [12]:
# only get the scripts name
all_scripts = []
for link in all_links:
    clean = remove_tags(str(link))
    if (len(clean) > 7) or ("." in clean):
        continue
    all_scripts.append(clean)

In [13]:
len(all_scripts)

501

In [ ]:
data = yf.download(
    tickers = all_scripts, 
    period = "1y",      # Use "1y" for one year
    group_by = 'ticker' # Organizes the multi-index by Ticker
)

data.head()

[*********************100%***********************]  501 of 501 completed


Ticker           ANET                                                    EW  \
Price            Open       High        Low      Close    Volume       Open   
Date                                                                          
2025-03-07  85.809998  86.430000  80.209999  83.360001  12894400  70.830002   
2025-03-10  80.000000  80.580002  76.000000  77.559998  15811700  69.480003   
2025-03-11  77.809998  81.139999  76.709999  79.129997  11291100  68.239998   
2025-03-12  82.440002  83.000000  79.339996  80.250000  11498000  69.110001   
2025-03-13  79.650002  80.760002  78.470001  80.139999   6973800  68.739998   

Ticker                                                ...       FITB  \
Price            High        Low      Close   Volume  ...       Open   
Date                                                  ...              
2025-03-07  71.529999  69.330002  70.330002  5207600  ...  38.341178   
2025-03-10  69.970001  68.180000  68.529999  4773200  ...  38.032441   
2025-03

In [ ]:
t = data.T


TypeError: DataFrame.query() got an unexpected keyword argument 'axis'

In [ ]:
idx = pd.IndexSlice
# Inside columns: [First Index Level, Second Index Level]
filtered_data = data.loc[:, idx[:, 'High']]
filtered_data[:5]

Ticker,ANET,EW,CBOE,AEE,HIG,WM,MSI,RJF,ISRG,NCLH,...,DECK,ADI,SYK,KMB,NI,URI,ATO,BAC,FITB,DRI
Price,High,High,High,High,High,High,High,High,High,High,...,High,High,High,High,High,High,High,High,High,High
Date,,,,,,,,,,,,,,,,,,,,,
2025-03-07,86.430000,71.529999,217.098867,96.548206,116.626801,226.528452,421.198103,141.863517,534.239990,20.100000,...,131.000000,223.253847,377.508745,140.596975,38.155299,625.363788,146.149234,40.812046,38.842871,189.685139
2025-03-10,80.580002,69.970001,218.053853,97.840318,117.511989,228.834150,417.854692,139.929106,503.910004,19.410000,...,124.800003,217.826631,373.634556,144.043666,37.978932,615.094395,149.661121,39.726457,38.592027,191.587926
2025-03-11,81.139999,69.019997,219.476393,97.165574,116.243233,224.863260,412.483344,138.774360,496.190002,19.010000,...,124.790001,214.123152,366.133907,141.037373,38.390466,603.665161,147.039431,39.765584,38.157863,189.364786
2025-03-12,83.000000,70.739998,210.225010,96.118579,115.544928,222.064890,412.503124,140.669308,519.510010,19.610001,...,126.349998,209.789268,367.709384,137.246014,38.527645,611.763743,145.024257,39.609107,37.569335,185.530100
2025-03-13,80.760002,68.919998,215.169023,96.451259,116.105542,220.783963,412.245929,140.205460,500.760010,19.320000,...,121.570000,205.031846,363.914460,135.752419,38.400261,610.207485,145.298188,39.305921,37.453558,185.617467


In [26]:
# Grab only the 'High' values across all tickers
high_df = data.xs('High', axis=1, level=1)
final_df = high_df.T
# final_df = high_df.stack(level=[0, 1]).unstack(level=0)
final_df

Date,2025-03-07,2025-03-10,2025-03-11,2025-03-12,2025-03-13,2025-03-14,2025-03-17,2025-03-18,2025-03-19,2025-03-20,...,2026-02-23,2026-02-24,2026-02-25,2026-02-26,2026-02-27,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06
Ticker,,,,,,,,,,,,,,,,,,,,,
ANET,86.430000,80.580002,81.139999,83.000000,80.760002,84.250000,85.739998,84.839996,85.000000,84.580002,...,131.634995,130.729996,133.779999,132.220001,133.580002,130.690002,126.830002,135.559998,139.479996,139.057999
EW,71.529999,69.970001,69.019997,70.739998,68.919998,69.660004,71.050003,71.080002,71.199997,71.849998,...,82.769997,83.029999,84.070000,85.919998,86.949997,87.320000,86.559998,85.809998,84.830002,83.080002
CBOE,217.098867,218.053853,219.476393,210.225010,215.169023,214.273714,217.735526,218.680553,218.849672,218.919300,...,292.856840,293.555110,294.273344,294.073829,303.119995,305.679993,305.000000,302.549988,301.739990,301.970001
AEE,96.548206,97.840318,97.165574,96.118579,96.451259,97.478701,98.711604,98.476771,98.300644,98.310427,...,112.000000,111.379997,112.000000,112.470001,113.440002,113.639999,113.099998,113.550003,112.879997,111.839996
HIG,116.626801,117.511989,116.243233,115.544928,116.105542,117.177576,119.016781,119.292165,117.954572,118.397157,...,143.008103,140.697995,140.628289,142.201556,141.285477,142.449997,141.539993,142.500000,142.149994,139.360001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
URI,625.363788,615.094395,603.665161,611.763743,610.207485,620.159715,622.221509,612.665787,626.533480,631.400535,...,904.630005,909.789978,904.500000,863.570007,845.890015,831.969971,842.859985,852.270020,857.750000,820.159973
ATO,146.149234,149.661121,147.039431,145.024257,145.298188,147.058999,149.123109,147.861155,147.812270,148.095950,...,182.690002,182.389999,182.479996,183.910004,187.820007,187.979996,186.460007,186.889999,186.509995,186.229996
BAC,40.812046,39.726457,39.765584,39.609107,39.305921,40.117666,40.851166,40.909851,41.760709,41.780274,...,52.960605,50.593984,51.478981,52.373920,51.121005,50.136570,50.226064,50.126625,50.146516,48.700001


In [27]:
final_df.to_csv('snp_500_dailyhigh.csv')